# Set Up and Load OpenAQ sensor data into BigQuery

-  Reads local CSV (bulk OpenAQ dataset).

-  Creates & populates the BigQuery table from scratch.

In [1]:
# 📌 1. Setup environment & imports

import os
from google.cloud import bigquery
import pandas as pd
from dotenv import load_dotenv

# Load environment variables (e.g., GOOGLE_APPLICATION_CREDENTIALS)
load_dotenv()

# Confirm key path
print("Google Credentials Path:", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

# Initialize BigQuery client
client = bigquery.Client()

Google Credentials Path: C:\\Users\\camer\\.gcp_keys\\openaq_data_loader.json


In [3]:
# 📌 2. Define your dataset and table names

PROJECT_ID = "openaq-data-pipeline-468404"
DATASET_ID = "openaq_ca"
TABLE_ID = "pm25_ca_sensors"
FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

print(f"BigQuery Project: {PROJECT_ID}")
print(f"BigQuery Dataset: {DATASET_ID}")
print(f"BigQuery Table: {TABLE_ID}")

BigQuery Project: openaq-data-pipeline-468404
BigQuery Dataset: openaq_ca
BigQuery Table: pm25_ca_sensors


In [4]:
# 📌 3. Create the dataset if it doesn’t exist

def create_dataset(dataset_id):
    dataset_ref = client.dataset(dataset_id)
    try:
        dataset = client.get_dataset(dataset_ref)
        print(f"Dataset {dataset_id} already exists.")
    except Exception:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"  # or your preferred region
        dataset = client.create_dataset(dataset)
        print(f"Created dataset {dataset_id}")

create_dataset(DATASET_ID)

Dataset openaq_ca already exists.


In [ ]:
# 📌 4. Load local CSV data to BigQuery table

CSV_PATH = "../data/openaq_pm25_ca_sensors.csv"

def load_csv_to_bq(csv_path, dataset_id, table_id):
    table_ref = client.dataset(dataset_id).table(table_id)

    job_config = bigquery.LoadJobConfig(
        autodetect=True,       # Auto-detect schema from data
        write_disposition="WRITE_TRUNCATE",  # Overwrite table if exists
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1   # Skip CSV header
    )

    with open(csv_path, "rb") as source_file:
        load_job = client.load_table_from_file(
            source_file,
            table_ref,
            job_config=job_config,
        )

    load_job.result()  # Wait for job to complete
    print(f"Loaded {load_job.output_rows} rows into {dataset_id}.{table_id}.")

load_csv_to_bq(CSV_PATH, DATASET_ID, TABLE_ID)

Loaded 59 rows into openaq_ca.pm25_ca_sensors.


In [6]:
# show table

query = f"""
SELECT *
FROM `{FULL_TABLE_ID}`
LIMIT 10
"""

query_job = client.query(query)
results = query_job.result()

for row in results:
    print(row)

Row((1488, 838, 'Universidad Autonoma', None, 'America/Tijuana', 157, 'MX', 'Mexico', 4, 'Unknown Governmental Organization', 119, 'AirNow', False, True, 32.6292, -115.4469, "{'utc': '2022-10-03T18:00:00Z', 'local': '2022-10-03T11:00:00-07:00'}"), {'sensor_id': 0, 'location_id': 1, 'location_name': 2, 'locality': 3, 'timezone': 4, 'country_id': 5, 'country_code': 6, 'country_name': 7, 'owner_id': 8, 'owner_name': 9, 'provider_id': 10, 'provider_name': 11, 'is_mobile': 12, 'is_monitor': 13, 'lat': 14, 'lon': 15, 'datetimeLast': 16})
Row((357, 214, 'MMFRA1001', None, 'America/Los_Angeles', 155, 'US', 'United States', 4, 'Unknown Governmental Organization', 119, 'AirNow', False, True, 39.482385, -121.221128, "{'utc': '2016-03-16T05:00:00Z', 'local': '2016-03-15T22:00:00-07:00'}"), {'sensor_id': 0, 'location_id': 1, 'location_name': 2, 'locality': 3, 'timezone': 4, 'country_id': 5, 'country_code': 6, 'country_name': 7, 'owner_id': 8, 'owner_name': 9, 'provider_id': 10, 'provider_name': 11,

In [7]:
# rename table

# from google.cloud import bigquery

# Initialize client
client = bigquery.Client()

# Define your dataset and table names
PROJECT_ID = "openaq-data-pipeline-468404"
DATASET_ID = "openaq_ca"
ORIGINAL_TABLE_ID = 'pm25_hourly_ca'
NEW_TABLE_ID = 'pm25_ca_hourly'

FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

print(f"BigQuery Project: {PROJECT_ID}")
print(f"BigQuery Dataset: {DATASET_ID}")
print(f"BigQuery Table: {TABLE_ID}")

# Full table IDs
ORIGINAL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.{ORIGINAL_TABLE_ID}"
NEW_TABLE = f"{PROJECT_ID}.{DATASET_ID}.{NEW_TABLE_ID}"

# Copy the table
job = client.copy_table(
    sources=ORIGINAL_TABLE,
    destination=NEW_TABLE,
    # Optionally, set write_disposition='WRITE_TRUNCATE' to overwrite if exists
)

job.result()  # Waits for the job to complete

print(f"Table {ORIGINAL_TABLE} copied to {NEW_TABLE}.")

# (Optional) Delete the original table if you want to complete the rename
client.delete_table(ORIGINAL_TABLE)
print(f"Deleted original table {ORIGINAL_TABLE}.")

BigQuery Project: openaq-data-pipeline-468404
BigQuery Dataset: openaq_ca
BigQuery Table: pm25_ca_sensors
Table openaq-data-pipeline-468404.openaq_ca.pm25_hourly_ca copied to openaq-data-pipeline-468404.openaq_ca.pm25_ca_hourly.
Deleted original table openaq-data-pipeline-468404.openaq_ca.pm25_hourly_ca.
